# Phase 6e Wave 4: Nonlinear Einstein Field Equations from ADW — Technical Notebook

Companion to `papers/paper42_nonlinear_efe/paper_draft.tex`.

**Lean module:** `lean/SKEFTHawking/NonlinearEFE.lean` (13 substantive theorems, 0 sorry, 0 new axioms; verified via `lake build`).

**Structure (mirrors paper §2–§6):**
1. Setup — Wave 1 + Wave 2 + Wave 3 + Phase 6a.1 inputs
2. Emergent vs bare-matter stress-energy traces + linear deviation channel
3. Trace-level EFE residual + Decision-Gate biconditional
4. Multi-channel post-Newtonian observables (deflection / precession / ringdown)
5. Cross-channel structural prediction (precession_dev = 2/3 × deflection_dev)
6. Higher-curvature correction at canonical backgrounds
7. Bundled tracked-Prop H_NonlinearEFEHolds
8. Figure: T_emerg vs matter + multi-channel observables
9. Lean theorem inventory

All numerical content imports from `src.nonlinear_efe` — no inline physics redefinition.


## 1. Setup — Wave 1 + Wave 2 + Wave 3 + Phase 6a.1 inputs

In [1]:
import numpy as np
from src.core.constants import NONLINEAR_EFE_PARAMS

print(f"α_ADW calibrated value     = {NONLINEAR_EFE_PARAMS['ALPHA_ADW_CALIBRATED']}")
print(f"α_ADW natural band         = "
      f"[{NONLINEAR_EFE_PARAMS['ALPHA_ADW_NATURAL_MIN']}, "
      f"{NONLINEAR_EFE_PARAMS['ALPHA_ADW_NATURAL_MAX']}]")
print(f"EFE residual tolerance     = {NONLINEAR_EFE_PARAMS['EFE_RESIDUAL_TOLERANCE']:.0e}")
print(f"T_emerg detection floor    = {NONLINEAR_EFE_PARAMS['T_EMERG_DEVIATION_DETECT_FLOOR']:.0e}")
print(f"VLBI deflection precision  = {NONLINEAR_EFE_PARAMS['DEFLECTION_OBS_RELATIVE_PRECISION']:.0e}")
print(f"MESSENGER perihelion       = {NONLINEAR_EFE_PARAMS['PERIHELION_OBS_RELATIVE_PRECISION']:.0e}")
print(f"GWTC-3 ringdown precision  = {NONLINEAR_EFE_PARAMS['RINGDOWN_OBS_RELATIVE_PRECISION']:.0e}")
print(f"Benchmark backgrounds      = {NONLINEAR_EFE_PARAMS['BENCHMARK_BACKGROUNDS']}")

α_ADW calibrated value     = 1.0
α_ADW natural band         = [0.1, 10.0]
EFE residual tolerance     = 1e-12
T_emerg detection floor    = 5e-03
VLBI deflection precision  = 3e-04
MESSENGER perihelion       = 1e-04
GWTC-3 ringdown precision  = 5e-02
Benchmark backgrounds      = ('Schwarzschild', 'de_Sitter', 'FLRW_radiation')


## 2. Emergent vs bare-matter stress-energy traces

Under the ADW substrate-rescaling model:
$$T_{\mathrm{emerg}} = \alpha_{\mathrm{ADW}} \cdot \rho_{\mathrm{ADW}}, \qquad T_{\mathrm{matter}} = \rho_{\mathrm{ADW}}.$$
The deviation channel
$$T_{\mathrm{emerg}} - T_{\mathrm{matter}} = (\alpha_{\mathrm{ADW}} - 1) \cdot \rho_{\mathrm{ADW}}$$
is **linear in $(\alpha_{\mathrm{ADW}} - 1)$** — the load-bearing observable signature.

In [2]:
from src.nonlinear_efe import compare_emergent_vs_matter, deviation_detectable_at_floor

rho = 1.0
print(f"{'α_ADW':>8s} {'T_emerg':>10s} {'T_matter':>10s} {'deviation':>12s} {'detectable':>12s}")
print("-" * 60)
for alpha in [0.5, 0.99, 1.0, 1.001, 1.05, 2.0]:
    rec = compare_emergent_vs_matter(rho_ADW=rho, alpha_ADW=alpha)
    detect = deviation_detectable_at_floor(rho_ADW=rho, alpha_ADW=alpha)
    print(f"{alpha:>8.3f} {rec.T_emerg:>10.4f} {rec.T_matter:>10.4f} "
          f"{rec.deviation:>12.4e} {detect!s:>12s}")

   α_ADW    T_emerg   T_matter    deviation   detectable
------------------------------------------------------------
   0.500     0.5000     1.0000  -5.0000e-01         True
   0.990     0.9900     1.0000  -1.0000e-02         True
   1.000     1.0000     1.0000   0.0000e+00        False
   1.001     1.0010     1.0000   1.0000e-03        False
   1.050     1.0500     1.0000   5.0000e-02         True
   2.000     2.0000     1.0000   1.0000e+00         True


## 3. Trace-level EFE residual + Decision-Gate biconditional

**Lean theorem `efeResidualTrace_eq_zero_iff_alpha_unity`:**

Under $G_N > 0$ and $\rho_{\mathrm{ADW}} \neq 0$,
$$\mathcal{R}_{\mathrm{EFE}}(G_N, \rho_{\mathrm{ADW}}, \alpha_{\mathrm{ADW}}) = 0
   \iff \alpha_{\mathrm{ADW}} = 1.$$

This is the nonlinear analogue of Wave 1's Decision Gate E.2 biconditional —
together they pin $\alpha_{\mathrm{ADW}} = 1$ as the unique algebraic locus
where both the linearized $a_2$ Einstein-Hilbert prefactor *and* the trace-level EFE close.

In [3]:
from src.nonlinear_efe import efe_residual_scan_over_alpha, efe_residual_at_dirac_balanced
import math

# Linearity demonstration
G_N, rho = 1.0, 1.0
alphas = np.array([0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 5.0])
scan = efe_residual_scan_over_alpha(G_N, rho, alphas)

print(f"{'α_ADW':>8s} {'residual':>15s} {'8π·G_N·ρ·(α-1)':>18s}")
print("-" * 50)
for alpha, resid in scan:
    expected = 8.0 * math.pi * G_N * rho * (alpha - 1.0)
    print(f"{alpha:>8.3f} {resid:>15.6e} {expected:>18.6e}")

# At the Dirac-Sakharov calibration: residual ≡ 0
print()
print(f"EFE residual at Dirac (Λ=10¹⁶, N_f=15, ρ=1, α=1): "
      f"{efe_residual_at_dirac_balanced(1e16, 15, 1.0, 1.0):.3e}")

   α_ADW        residual     8π·G_N·ρ·(α-1)
--------------------------------------------------
   0.500   -1.256637e+01      -1.256637e+01
   0.750   -6.283185e+00      -6.283185e+00
   1.000    0.000000e+00       0.000000e+00
   1.250    6.283185e+00       6.283185e+00
   1.500    1.256637e+01       1.256637e+01
   2.000    2.513274e+01       2.513274e+01
   5.000    1.005310e+02       1.005310e+02

EFE residual at Dirac (Λ=10¹⁶, N_f=15, ρ=1, α=1): 0.000e+00


## 4. Multi-channel post-Newtonian observables

**Three observable channels under the ADW $\alpha_{\mathrm{ADW}}$ rescaling:**

| Channel | Functional form | Source |
|---------|-----------------|--------|
| Light deflection | $\delta\theta_{\mathrm{ADW}}/\delta\theta_{\mathrm{GR}} = \alpha_{\mathrm{ADW}}$ | Will 2018 §4.1 |
| Perihelion precession | $(2 + 2\gamma - \beta)/3 = (2\alpha_{\mathrm{ADW}} + 1)/3$ at $\gamma = \alpha, \beta = 1$ | Will 2018 Eq. 4.31 |
| Ringdown frequency | $\omega_{\mathrm{ADW}}/\omega_{\mathrm{GR}} = \alpha_{\mathrm{ADW}}$ | Berti-Cardoso-Starinets 2009 |

Note the perihelion functional form is **non-trivially distinct** from the linear-in-$\alpha$ deflection and ringdown channels.

In [4]:
from src.nonlinear_efe import cross_channel_signature_table

print(f"{'channel':>15s} {'α=1.0':>12s} {'α=1.05':>12s} {'α=1.5':>12s}")
print("-" * 60)
sigs_at_calib = {s.channel: s for s in cross_channel_signature_table(1.0)}
sigs_at_5pct = {s.channel: s for s in cross_channel_signature_table(1.05)}
sigs_at_50pct = {s.channel: s for s in cross_channel_signature_table(1.5)}
for ch in ["deflection", "precession", "ringdown"]:
    print(f"{ch:>15s} "
          f"{sigs_at_calib[ch].ratio:>12.6f} "
          f"{sigs_at_5pct[ch].ratio:>12.6f} "
          f"{sigs_at_50pct[ch].ratio:>12.6f}")

        channel        α=1.0       α=1.05        α=1.5
------------------------------------------------------------
     deflection     1.000000     1.050000     1.500000
     precession     1.000000     1.033333     1.333333
       ringdown     1.000000     1.050000     1.500000


## 5. Cross-channel structural prediction

**Lean theorem `precession_dev_eq_two_thirds_deflection_dev`:**

$$\frac{\delta\varphi_{\mathrm{ADW}}}{\delta\varphi_{\mathrm{GR}}} - 1
   = \frac{2}{3}\left(\frac{\delta\theta_{\mathrm{ADW}}}{\delta\theta_{\mathrm{GR}}} - 1\right)$$

The **multi-observation testable claim**: any campaign measuring deflection and perihelion
deviations must find them in the 3:2 ratio (perihelion = 2/3 of deflection).
A different ratio falsifies the ADW model.

In [5]:
print(f"{'α_ADW':>8s} {'defl_dev':>12s} {'prec_dev':>12s} {'prec/defl':>12s}")
print("-" * 50)
for alpha in [0.5, 0.9, 1.0, 1.1, 1.5, 2.0, 5.0]:
    defl = (alpha - 1.0)
    prec = (2.0 * alpha + 1.0) / 3.0 - 1.0
    ratio = prec / defl if defl != 0 else float('nan')
    print(f"{alpha:>8.3f} {defl:>12.6e} {prec:>12.6e} {ratio:>12.6f}")

   α_ADW     defl_dev     prec_dev    prec/defl
--------------------------------------------------
   0.500 -5.000000e-01 -3.333333e-01     0.666667
   0.900 -1.000000e-01 -6.666667e-02     0.666667
   1.000 0.000000e+00 0.000000e+00          nan
   1.100 1.000000e-01 6.666667e-02     0.666667
   1.500 5.000000e-01 3.333333e-01     0.666667
   2.000 1.000000e+00 6.666667e-01     0.666667
   5.000 4.000000e+00 2.666667e+00     0.666667


## 6. Higher-curvature correction at canonical backgrounds

In [6]:
from src.nonlinear_efe import (
    schwarzschild_horizon_background,
    de_sitter_background,
    flrw_radiation_background,
)
from src.core.formulas import higher_curvature_correction_at_background

bgs = [
    schwarzschild_horizon_background(),
    de_sitter_background(),
    flrw_radiation_background(),
]

print(f"{'background':>22s} {'R':>6s} {'R²':>8s} {'R_μν²':>8s} {'R_μνρσ²':>10s} "
      f"{'a4 correction (N_f=24)':>30s}")
print("-" * 90)
for bg in bgs:
    correction = higher_curvature_correction_at_background(
        24, bg.R_sq, bg.Ricci_sq, bg.Riemann_sq
    )
    print(f"{bg.name:>22s} {bg.R:>6.1f} {bg.R_sq:>8.1f} "
          f"{bg.Ricci_sq:>8.1f} {bg.Riemann_sq:>10.1f} {correction:>30.6e}")

            background      R       R²    R_μν²    R_μνρσ²         a4 correction (N_f=24)
------------------------------------------------------------------------------------------
 Schwarzschild_horizon    0.0      0.0      0.0        3.0                  -2.533030e-03
          de_Sitter_H1   12.0    144.0     36.0       24.0                  -5.319362e-02
     FLRW_radiation_H1    0.0      0.0     12.0       12.0                  -4.221716e-03


## 7. Bundled tracked-Prop `H_NonlinearEFEHolds`

**Lean def `H_NonlinearEFEHolds(Λ, N_f, ρ_ADW, α_ADW)`:**

$$\begin{aligned}
H_{\mathrm{NonlinearEFEHolds}} :\equiv\
&(C_1)\ \mathcal{R}_{\mathrm{EFE}} = 0 \\
&\land (C_2)\ H_{\mathrm{HigherCurvatureWithinObservationalBounds}}(10^{59}) \\
&\land (C_3)\ H_{\mathrm{NonlinearDiffInvariance}}(\mathrm{diracCoefBundle}\;N_f, N_f)
\end{aligned}$$

Each conjunct invokes a *distinct* Wave by name — substantive cross-bridge consumer.

In [7]:
from src.core.formulas import nonlinear_efe_holds

# Bundle holds at calibration
print(f"H_NonlinearEFEHolds at calibration (α=1.0):     "
      f"{nonlinear_efe_holds(1e16, 15, 1.0, 1.0)}")

# Bundle fails off calibration
for alpha in [1.001, 1.5, 2.0]:
    holds = nonlinear_efe_holds(1e16, 15, 1.0, alpha)
    print(f"H_NonlinearEFEHolds at α={alpha:5.3f}:                   "
          f"{holds}")

# All SM-relevant N_f satisfy the bundle at α=1
print()
print("SM-relevant N_f sweep at α=1:")
for N_f in [1, 6, 15, 24, 27]:
    holds = nonlinear_efe_holds(1e16, N_f, 1.0, 1.0)
    print(f"  N_f={N_f:>3d}: {holds}")

H_NonlinearEFEHolds at calibration (α=1.0):     True
H_NonlinearEFEHolds at α=1.001:                   False
H_NonlinearEFEHolds at α=1.500:                   False
H_NonlinearEFEHolds at α=2.000:                   False

SM-relevant N_f sweep at α=1:
  N_f=  1: True
  N_f=  6: True
  N_f= 15: True
  N_f= 24: True
  N_f= 27: True


## 8. Figure: T_emerg vs matter + multi-channel observables

In [8]:
from src.core.visualizations import fig_T_emerg_vs_matter
fig = fig_T_emerg_vs_matter()
fig.show()

## 9. Lean theorem inventory

The 13 substantive theorems of `NonlinearEFE.lean`:

| # | Theorem | Content |
|---|---------|---------|
| 1 | `emergentStressEnergyTrace_eq_matter_iff_alpha_unity` | T_emerg = T_matter ⇔ α=1 (under ρ ≠ 0) |
| 2 | `emergentStressEnergyTrace_minus_matter_eq` | Linear deviation channel: T_emerg − T_matter = (α−1)ρ |
| 3 | `emergentStressEnergyTrace_pos_iff_alpha_pos` | Sign-positivity bridge to Vergeles α>0 |
| 4 | `efeResidualTrace_at_alpha_one` | Residual = 0 at calibration α=1 |
| 5 | `efeResidualTrace_eq_zero_iff_alpha_unity` | **MAIN.** Decision-Gate biconditional |
| 6 | `efeResidualTrace_at_dirac_calibration_vanishes` | Substantive cross-bridge to Wave 1 + 6a.1 |
| 7 | `deflectionRatio_minus_one_eq` | Linearity of the deflection deviation |
| 8 | `precessionRatio_eq_one_iff_alpha_unity` | Non-trivial PPN biconditional (linarith) |
| 9 | `precession_dev_eq_two_thirds_deflection_dev` | **Cross-channel structural claim** |
| 10 | `deflectionRatio_deviation_exceeds_VLBI_floor` | Quantitative Lean ↔ Python tolerance bridge |
| 11 | `higherCurvatureCorrection_abs_bound` | Substantive cross-bridge to Wave 2 pulsar bound |
| 12 | `dirac_H_NonlinearEFEHolds_at_alpha_one` | Bundled-Prop discharge witness |
| 13 | `perturbed_alpha_not_H_NonlinearEFEHolds` | Bundled-Prop discriminator falsifier |

**Decision Gate verdict:** Wave 4's correctness-push biconditional pins α_ADW = 1 as the unique calibration value at which the trace-level nonlinear EFE closes. Phase 6e Wave 5 (microscopic-to-macroscopic coefficient match) and Wave 6 (Einstein-Cartan extension) consume `H_NonlinearEFEHolds` as a starting input.
